Segunda tarefa do Job:
- Criar o schema da camada Silver, e cria as tabelas Delta sobre os arquivos "csv" brutos da camada Bronze.

In [0]:
from pyspark.sql import functions as F

try:
    spark.sql('CREATE SCHEMA IF NOT EXISTS databricks_cnpj_data_lakehouse.silver')

    ano_mes_pasta = dbutils.jobs.taskValues.get(taskKey="ingestao_bronze", key="ano_mes_pasta", debugValue='2026_08')
    if not ano_mes_pasta:
        raise ValueError("ano_mes_pasta não retornado pela task 'ingestao_bronze' (taskValues vazio ou key ausente)")

    arquivos = ['Cnaes.csv','Empresas*.csv','Estabelecimentos*.csv','Motivos.csv','Municipios.csv','Naturezas.csv','Paises.csv','Qualificacoes.csv','Simples.csv','Socios*.csv']    

    for arquivo in arquivos:
        nome_base = arquivo[:-4].replace('*', '')
        caminho = f"/Volumes/databricks_cnpj_data_lakehouse/bronze/{ano_mes_pasta}/{arquivo}"

        print(f"Processando: {caminho})")

        df_raw = (
            spark.read
                .option("sep", ";")
                .option("header", "false")
                .option("encoding", "iso-8859-1")
                .option("quote", "\"")
                .option("escape", "\"")
                .option("multiLine", "true")
                .csv(caminho)
                .withColumn("dt_processamento", F.current_timestamp())
        )

        df_raw.write.mode("overwrite").format("delta").saveAsTable(f"databricks_cnpj_data_lakehouse.silver.raw_{nome_base}")        
        
except Exception as e:
    print(f"FALHA ao processar {e}")
    raise